# Notebook 03 — Cálculo de Impedâncias (Tempos de Viagem Multimodal)

**Projeto:** Acessibilidade Geográfica às UBS de Teresina — Pipeline AE2SFCA  
**Autor:** Felipe Ramos Dantas  
**Programa:** MAPEPROF / IFPI  

---
## Objetivo
Com a malha viária 100% classificada pela Inteligência Artificial (Notebook 02.2), esta etapa calcula a **impedância** de cada segmento da rede. A impedância, na ciência de redes, é o "custo" de atravessar uma aresta. Para a metodologia AE2SFCA, este custo é medido em **minutos**.

O cálculo será feito para dois modais de transporte:
1. **Veículo Automotor (Carro):** Utilizando velocidades regulamentares por hierarquia viária.
2. **Pedestre (Caminhada):** Assumindo uma velocidade média humana em ambiente urbano (aprox. 4.8 km/h).

**Fórmula Fundamental:** $t = \left( \frac{d}{v} \right) \times 60$  
*(Onde $t$ é o tempo em minutos, $d$ é a distância em km, e $v$ é a velocidade em km/h)*

In [5]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd

warnings.filterwarnings("ignore")

PROCESSED_PATH = Path('../dados/processados')

print("🔄 Carregando a malha viária enriquecida pela GNN...")
gdf_roads = gpd.read_parquet(PROCESSED_PATH / "zonaUrbanaReal_5km_road_gnn_imputed.parquet")

print(f"✅ Malha carregada: {len(gdf_roads):,} segmentos disponíveis para roteamento.")

🔄 Carregando a malha viária enriquecida pela GNN...
✅ Malha carregada: 52,431 segmentos disponíveis para roteamento.


In [6]:
# ── CÉLULA EXTRA (Início do NB 03): Filtro de Roteabilidade ─────────────────

# Remove vias fisicamente isoladas de carros e vias restritas/industriais
classes_excluidas = ['pedestrian', 'footway', 'path', 'steps', 'service']

qtd_antes = len(gdf_roads)
gdf_roads = gdf_roads[~gdf_roads['class'].isin(classes_excluidas)].copy()
qtd_depois = len(gdf_roads)

print(f"Filtro aplicado: {qtd_antes - qtd_depois} segmentos não-roteáveis removidos.")

Filtro aplicado: 5996 segmentos não-roteáveis removidos.


In [2]:
# ── 1. Matriz de Velocidades por Hierarquia (Carros) ────────────────────────
# Estes valores (km/h) baseiam-se em diretrizes de trânsito urbano padrão.
# Você pode ajustá-los de acordo com o plano diretor de Teresina, se necessário.
SPEED_DICT = {
    'motorway': 80,       # Vias expressas
    'trunk': 70,          # Vias de trânsito rápido
    'primary': 60,        # Avenidas arteriais principais
    'secondary': 50,      # Avenidas coletoras secundárias
    'tertiary': 40,       # Vias coletoras de bairro
    'residential': 30,    # Ruas locais (tráfego calmo)
    'living_street': 20,  # Zonas de convivência/compartilhadas
    'pedestrian': 0.1,    # Carro não passa (Penalidade máxima)
    'footway': 0.1,       # Carro não passa
    'track': 20,          # Estradas de terra/acesso
    'service': 20,        # Vias de serviço/estacionamento
}

# Velocidade média de um pedestre adulto em ambiente urbano (km/h)
VELOCIDADE_PEDESTRE_KMH = 4.8 

print("Tabela de velocidades teóricas definida.")

Tabela de velocidades teóricas definida.


In [7]:
# ── 2. Cálculo da Velocidade Final do Segmento ──────────────────────────────
print("🚗 Calculando dinâmicas de velocidade...")

# Passo A: Aplicamos a velocidade teórica baseada na classe prevista pela GNN
gdf_roads['velocidade_teorica'] = gdf_roads['class'].map(SPEED_DICT)

# Preenchimento de segurança caso alguma classe muito exótica tenha escapado
gdf_roads['velocidade_teorica'].fillna(30, inplace=True)

# FATOR DE FRICÇÃO URBANA (Adaptação metodológica)
# Reduzimos a velocidade de fluxo livre em 30% para simular paradas em cruzamentos, 
# lombadas e semáforos, aderindo à literatura de tráfego urbano.
FATOR_FRICCAO = 0.70

# Aplicamos o atrito sobre a velocidade teórica
gdf_roads['velocidade_friccao'] = gdf_roads['velocidade_teorica'] * FATOR_FRICCAO

# Passo B: Lógica de Fusão de Velocidade
# Se a Overture já fornecia a 'velocidade_max' (do NB 02.1), confiamos nela.
# Caso contrário, usamos a teórica.
gdf_roads['velocidade_final_carro'] = np.where(
    gdf_roads['velocidade_max'].notna(), 
    gdf_roads['velocidade_max'] * FATOR_FRICCAO, 
    gdf_roads['velocidade_teorica']
)

# ── 3. Cálculo do Tempo de Viagem (Impedância em Minutos) ───────────────────
print("⏱️ Calculando as impedâncias (custo em minutos) para o grafo...")

# A distância foi calculada em metros no NB 02.2. Convertendo para km para a fórmula.
distancia_km = gdf_roads['length_m'] / 1000.0

# Tempo Veículo Automotor (Minutos)
gdf_roads['tempo_carro_min'] = (distancia_km / gdf_roads['velocidade_final_carro']) * 60

# Tempo Pedestre (Minutos)
gdf_roads['tempo_pedestre_min'] = (distancia_km / VELOCIDADE_PEDESTRE_KMH) * 60

t = gdf_roads['tempo_carro_min']
assert np.isfinite(t).all() and (t >= 0).all(), \
    "Há tempos negativos, infinitos ou ausentes; só esses impedem o roteamento."

print(f"Tempo carro: média {t.mean():.2f} min "
      f"(mín {t.min():.3f}, máx {t.max():.1f}) | segmentos com tempo zero: {(t == 0).sum()}")

print("✅ Tempos de viagem calculados com sucesso!")
print(f"   -> Tempo médio p/ atravessar uma rua (Carro): {gdf_roads['tempo_carro_min'].mean():.2f} min")
print(f"   -> Tempo médio p/ atravessar uma rua (A pé): {gdf_roads['tempo_pedestre_min'].mean():.2f} min")

🚗 Calculando dinâmicas de velocidade...
⏱️ Calculando as impedâncias (custo em minutos) para o grafo...
Tempo carro: média 0.18 min (mín 0.000, máx 14.3) | segmentos com tempo zero: 29
✅ Tempos de viagem calculados com sucesso!
   -> Tempo médio p/ atravessar uma rua (Carro): 0.18 min
   -> Tempo médio p/ atravessar uma rua (A pé): 1.15 min


In [8]:
# ── 4. Validação e Exportação ───────────────────────────────────────────────
# Removemos colunas que não servirão mais para o cálculo do AE2SFCA
colunas_para_remover = ['velocidade_teorica']
gdf_roads.drop(columns=colunas_para_remover, inplace=True, errors='ignore')

# Salvamento final
caminho_final = PROCESSED_PATH / "zonaUrbanaReal_5km_road_routed.parquet"
gdf_roads.to_parquet(caminho_final, index=False)

print(f"💾 Grafo finalizado com impedâncias de roteamento salvo em: {caminho_final}")
print("   (Pronto para o algoritmo de Caminho Mais Curto / Isócronas - Notebook 04)")

💾 Grafo finalizado com impedâncias de roteamento salvo em: ..\dados\processados\zonaUrbanaReal_5km_road_routed.parquet
   (Pronto para o algoritmo de Caminho Mais Curto / Isócronas - Notebook 04)
